<a href="https://colab.research.google.com/github/ysasson-portfolio/text-analytics-spring-2026/blob/main/assignment_5/notebooks/Yarden_Sasson_A5_OptionB_Job_Fit_Starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 5 — Option B: Job Fit Analyzer
## BSAN 6200: Text Mining & Social Media Analytics — Spring 2026

**Student Name:** Yarden Sasson
**Date:** May 13, 2026  
**Option:** B — Job Fit Analyzer  
**API Path:** Paid

---

### Table of Contents
1. [Setup and Imports](#1-setup)
2. [Load Job Descriptions and Resume](#2-loading)
3. [Text Chunking](#3-chunking)
4. [Embedding and Vector Store](#4-embedding)
5. [Analysis Prompts and Chain](#5-analysis)
6. [Zero-shot vs. Few-shot Comparison](#6-comparison)
7. [Evaluation](#7-evaluation)

> **Reminder:** The Streamlit app is a separate file (`streamlit_app.py`). This notebook builds and tests the analysis pipeline.  
> See the Option B Implementation Guide for detailed step requirements.

---
<a id="1-setup"></a>
## 1. Setup and Imports

Install required packages and load your API key from a `.env` file.  
**Do NOT hardcode API keys in this notebook.**

Suggested packages: `langchain`, `langchain-openai` or `langchain-community`, `chromadb` or `faiss-cpu`, `pypdf`, `python-dotenv`, `pandas`, `sentence-transformers` (free path)

In [84]:
# ── Install packages (uncomment as needed) ──
!pip install langchain langchain-openai chromadb pypdf python-dotenv sentence-transformers
!pip install -q langchain langchain-community
# ── Load API keys from .env ──
import os
import pandas as pd
from dotenv import load_dotenv
from langchain_core.documents import Document
import requests

# ── Your imports below ──

In [85]:
#Clone the Repository (only needs to happen once)
!git clone https://github.com/ysasson-portfolio/text-analytics-spring-2026.git

fatal: destination path 'text-analytics-spring-2026' already exists and is not an empty directory.


In [86]:
#If the github gets updated and I want the latest changes without re-cloning
%cd /content/text-analytics-spring-2026
!git pull

/content/text-analytics-spring-2026
Already up to date.


---
<a id="2-loading"></a>
## 2. Load Job Descriptions and Resume

**Required:**
- 10+ JD files in `data/job_descriptions/` (each as a separate .txt or .pdf)
- Your resume in `data/resume/`
- A metadata file `data/jd_metadata.csv` with columns: filename, company, title, source_url, date_collected

Print: number of JDs loaded, number of resume docs, and preview content from each.

In [87]:
# ── Load JD metadata ──
metadata_df = pd.read_csv("https://raw.githubusercontent.com/ysasson-portfolio/text-analytics-spring-2026/refs/heads/main/assignment_5/data/jd_metadata.csv")

print(metadata_df)

                                           Job Title  \
0                      Business Intelligence Analyst   
1  Business Intelligence Analyst, Sports - Brand ...   
2                      Business Intelligence Analyst   
3                                   Business Analyst   
4                                   Business Analyst   
5                                   Business Analyst   
6                                 Business Analyst I   
7                Sr. Analyst, Strategy and Analytics   
8                                 Strategy Associate   
9                                  Manager, Strategy   

                                    Company  \
0                             Guitar Center   
1                   Creative Artists Agency   
2  Los Angeles Tourism and Convention Board   
3             Red Bull Distribution Company   
4                                   Hadrian   
5                        Polestar Analytics   
6                  Skyworks Solutions, Inc.   
7      

In [88]:
#Use the folder path that is directly connected to the job description folder
folder_path = "/content/text-analytics-spring-2026/assignment_5/data/job_descriptions/"

#Empty list to store the descriptions and the metadata
job_descriptions=[]

for index, row in metadata_df.iterrows():
    #Pull the file name from the metadata dataframe
    filename = row["File Name"] + ".txt"
    #Create the final path using the folder path and file name
    file_path = os.path.join(folder_path, filename)

    #Create the document with the following metadata from the dataframe
    document = Document(
        page_content=open(file_path, "r", encoding="utf-8").read(),
        metadata={
            "File Name": row["File Name"],
            "Company": row["Company"],
            "Job Title": row["Job Title"],
            "Source URL": row["Source URL"],
            "Date Collected": row["Date Collected"],
            "Document Type": "Job Description"
        }
    )
    #Add the job description to the empty list
    job_descriptions.append(document)


print(f"Number of JDs loaded: {len(job_descriptions)}")

Number of JDs loaded: 10


In [89]:
#Show the sample of the metadata and the job description generated
print(job_descriptions[0].metadata)
print(job_descriptions[0].page_content)

{'File Name': 'Business Inteligence Analyst-Guitar Center', 'Company': 'Guitar Center', 'Job Title': 'Business Intelligence Analyst', 'Source URL': 'https://www.linkedin.com/jobs/collections/recommended/?currentJobId=4384414218&origin=JYMBII_IN_APP_NOTIFICATION&originToLandingJobPostings=4392357648%2C4407124320', 'Date Collected': '4/28/2026', 'Document Type': 'Job Description'}
About the Role:  
At Guitar Center, the Data Team is deploying data to inform and empower the company with insight, to drive customer success and business value. We are looking for a Business Intelligence Analyst to help advance that vision as part of the Business Intelligence team, which is part of the larger central Data Organization. You will work with stakeholders and data peers to create a vibrant ecosystem that enables data self-service across the company. This is a chance to affect millions of musicians at the United States' largest musical retailer, and your success as part of a world-class analytics or

In [90]:
#Use the folder path that is directly connected to the resume
resume_path = "https://raw.githubusercontent.com/ysasson-portfolio/text-analytics-spring-2026/refs/heads/main/assignment_5/data/resume/resume.txt"

resume_text= requests.get(resume_path).text

#Create the document with the following metadata that we supplied
resume = Document(
    page_content=resume_text,
    metadata_df={
        "File Name": "resume.txt",
        "Document Type": "Resume"}
)

print("Resume has been loaded")
print(resume.page_content)

Resume has been loaded
Yarden Sasson


EDUCATION

Loyola Marymount University                                                        
Masters of Science in Business Analytics                                                                                         Class of 2026
•	Relevant Coursework: Data Management for Business Intelligence, Introduction to Machine Learning, Strategic Integration Analytics
•	Honors Society: Beta Gamma Sigma

University of California, Los Angeles (UCLA)                                                        
Bachelors of Science in Cognitive Science with a Specialization in Computing                               Class of 2020
•	Relevant Coursework: Advanced Topics in MATLAB Programming for Behavioral Sciences, Science of Language, and Neural Networks
•	Activities: Den Operations Club, UCLA’s The Den, UCLA Hillel

WORK EXPERIENCE

Israel Economic and Trade Mission to the West Coast                                                              
Head of Inn

In [91]:
# ── Preview sample content ──


---
<a id="3-chunking"></a>
## 3. Text Chunking

Split your documents into chunks.  
**Required:** Try at least 2 chunking strategies, compare them quantitatively, and justify your final choice.

**Hint:** JDs often have natural sections (Requirements, Responsibilities, Qualifications). Consider whether your splitter respects these boundaries.

In [92]:
#Combine all documents so we can chunk them effectively
every_document= job_descriptions + [resume]

In [93]:
# ── Strategy 1 ──
def chunk_text(text, chunk_size=300, overlap=50):
    """Split text into overlapping chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk.strip())
        start += chunk_size - overlap
    return [c for c in chunks if len(c) > 20]  # skip tiny fragments

fixed_chunks = []
#Apply the strategy to each document and cumulatively counting the number of chunks while also
for doc in every_document:
    chunks = chunk_text(doc.page_content, chunk_size=400, overlap=100)

    for i, chunk in enumerate(chunks):
        fixed_chunks.append({
            "Text": chunk,
            "File Name": doc.metadata.get("File Name", ""),
            "Company": doc.metadata.get("Company", ""),
            "Job Title": doc.metadata.get("Job Title", ""),
            "Document Type": doc.metadata.get("Document Type", ""),
            "Chunk_id": i,
            "Strategy": "Fixed Size"
        })


print("Strategy 1 chunks:", len(fixed_chunks))
print(f"\nChunks per source:")
from collections import Counter
for job_title, count in Counter(c.get("Company", "Resume") or "Resume"
        for c in fixed_chunks).items():
    print(f"  {job_title}: {count}")


Strategy 1 chunks: 155

Chunks per source:
  Guitar Center: 10
  Creative Artists Agency: 10
  Los Angeles Tourism and Convention Board: 27
  Red Bull Distribution Company: 13
  Hadrian: 18
  Polestar Analytics: 13
  Skyworks Solutions, Inc.: 8
  SoFi Stadium and Hollywood Park: 24
  Cedars Sinai: 12
  Paramount: 8
  Resume: 12


In [94]:
# ── Strategy 2 ──
def chunk_text_by_sentences(text, chunk_size=400, overlap_words=20):
    """Split text into chunks that respect sentence boundaries."""
    import re
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())

    chunks = []
    current_chunk = ""

    for sentence in sentences:
        if len(current_chunk) + len(sentence) > chunk_size and current_chunk:
            chunks.append(current_chunk.strip())
            # Keep overlap by taking the end of the current chunk
            words = current_chunk.split()
            overlap_text = " ".join(words[-overlap_words:]) if len(words) > overlap_words else current_chunk
            current_chunk = overlap_text + " " + sentence
        else:
            current_chunk += (" " if current_chunk else "") + sentence

    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    return chunks

sentence_chunks = []

#Apply the strategy to each document while cumulatively counting the chunks and storing the relavent metadata within each chunk
for doc in every_document:
    chunks = chunk_text_by_sentences(doc.page_content, chunk_size=400, overlap_words=10)

    for i, chunk in enumerate(chunks):
        sentence_chunks.append({
            "Text": chunk,
            "File Name": doc.metadata.get("File Name", ""),
            "Company": doc.metadata.get("Company", ""),
            "Job Title": doc.metadata.get("Job Title", ""),
            "Document Type": doc.metadata.get("Document Type", ""),
            "Chunk_id": i,
            "Strategy": "Sentence Aware"
        })

print("Strategy 2 chunks:", len(sentence_chunks))
print(f"\nChunks per source:")
from collections import Counter
for job_title, count in Counter(c.get("Company", "Resume") or "Resume"
        for c in sentence_chunks).items():
    print(f"  {job_title}: {count}")

Strategy 2 chunks: 143

Chunks per source:
  Guitar Center: 11
  Creative Artists Agency: 9
  Los Angeles Tourism and Convention Board: 29
  Red Bull Distribution Company: 14
  Hadrian: 9
  Polestar Analytics: 15
  Skyworks Solutions, Inc.: 7
  SoFi Stadium and Hollywood Park: 26
  Cedars Sinai: 14
  Paramount: 4
  Resume: 5


### Chunking Decision

Looking at the overall size of the job descriptions and the resume, we can see that the size in terms of length varies. There are some such as the Paramount and the Skyworks Solutions job descriptions are shorter while the LA Tourism and Convention Board and SoFi Stadium and Hollywood Park have longer descriptions. This is why we need to have a cluster size that is small enough to maintain the meaning of the text along with the context, while having a large enough cluster size that produces multiple clusters.

After messing around with the cluster sizes to the number of clusters that were produced, I believed that the best results were produced at a cluster size of 400. Even though some of the job descriptions ended up with a higher number of clusters, there are enough clusters for it to perform well enough with the LLM model that will eventually be chosen.

**Which strategy did you choose? Why?**

In the end, I chose a sentence based chunking strategy (Strategy 2). The sentence based method produced a smaller amount of clusters. The fixed-chunk method produced a total of 155 based on the all job descriptions and the resume, while the sentence based method produced 143 clusters. That is a difference of 9.09% between the two. I also decided to use this method because it will most likely acheive better context within the chunks. Since this strategy considers the whole word in the overlap instead of the character count, it will not end the cluster in the middle of word giving us words that do not exist. When looking at the amount of overlap, I wanted to make sure that there would be enough words to make sure that context and semantic meaning would be understood by the model. With shorter job descriptions I wanted to make sure that the overlap would not be as drastic to lower the number of clusters or give it too much information per cluster that would cause the model to overfit.

**Final settings (chunk_size, overlap):**
Chunk Size: 400 characters
Overlap Words: 10 words

---
<a id="4-embedding"></a>
## 4. Embedding and Vector Store

Embed your chunks and store them in a vector database (ChromaDB or FAISS).

**Paid path:** OpenAI `text-embedding-3-small`  
**Free path:** `sentence-transformers/all-MiniLM-L6-v2`

After creating the store, run a test similarity search to verify it works.

In [95]:
from sentence_transformers import SentenceTransformer
import chromadb

# Load embedding model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# Check embedding size
sample = embed_model.encode("What job is this resume a good fit for?")
print(f"Embedding dimensions: {len(sample)}")
print(f"First 10 values: {sample[:10].tolist()}")

# Create ChromaDB client
chroma_client = chromadb.Client()

# Create or reset collection
collection = chroma_client.get_or_create_collection(
    name="job_fit_sentence_chunks",
    metadata={"hnsw:space": "cosine"}
)

# Pull text and metadata from your sentence-based chunks
documents = [c["Text"] for c in sentence_chunks]

metadatas = [
    {
        "job_title": c.get("Job Title", "Resume") or "Resume",
        "company": c.get("Company", "Resume") or "Resume",
        "source": c.get("Job Title", "Resume") or "Resume"
    }
    for c in sentence_chunks
]

ids = [f"sentence_chunk_{i}" for i in range(len(sentence_chunks))]

# Create embeddings yourself using SentenceTransformer
embeddings = embed_model.encode(documents).tolist()

# Add chunks to ChromaDB
collection.add(
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas,
    ids=ids
)

print(f"Vector store created with {collection.count()} sentence-based chunks")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dimensions: 384
First 10 values: [-0.11932417005300522, 0.055823106318712234, -0.007771510165184736, 0.033555809408426285, -0.012139878235757351, 0.055921223014593124, -0.0020768812391906977, -0.004593969322741032, -0.1080927774310112, -0.060754548758268356]
Vector store created with 143 sentence-based chunks


In [96]:
# ── Verify: run a test similarity search ──

def search(query, k=3):
    """Search the vector store and return top-k results."""
    results = collection.query(
        query_texts=[query],
        n_results=k
    )
    # ChromaDB returns nested lists, so we unpack
    docs = results["documents"][0]
    sources = [m["source"] for m in results["metadatas"][0]]
    distances = results["distances"][0]
    return list(zip(docs, sources, distances))


# ── Test: technology question ──
query = "Which jobs require sql?"
print(f'Query: "{query}"\n')

for i, (text, source, dist) in enumerate(search(query, k=3)):
    print(f"Result {i+1} [Source: {source}] (distance: {dist:.3f}):")
    print(f"  {text[:400]}")
    print()

Query: "Which jobs require sql?"

Result 1 [Source: Sr. Analyst, Strategy and Analytics] (distance: 0.472):
  visualization tools (Tableau and Google Looker expertise is a plus). Advanced proficiency with ETL and other data workflow tools (Alteryx, Matillion, etc). Proficient with common data languages (SQL, Python, R, etc). Experience using workflows for automation purposes is a plus. Proficiency with Ticketmaster / Archtics Data Platform preferred.

Result 2 [Source: Business Intelligence Analyst] (distance: 0.504):
  of relevant experience is required. High school diploma or equivalent. Must be willing to partake in a comprehensive background check including a drug test in accordance with applicable laws. Proficiency with Domo or other Business Intelligence tools (i.e., Tableau, Power BI). Proficiency in SQL and Python programming. Data architecture design and data preparation experience.

Result 3 [Source: Business Analyst] (distance: 0.512):
  Business, Economics, Information Syst

After embedding the infomration, I ran a similarity test by asking "What jobs require SQL?". After looking at the responses I see that responses have a moderate score after using a cosine similarity score. The responses all showed that a SQL proficiency was necessary for the job.

The first results produced a stronger than moderate result with a score 0.472. It was the one where SQL appears earlier and requires a strong proficiency. The second result showed a strong profieciency as well later on in the description. This had a slightly worse score than the first on at 0.504. The third one did not identify SQL as a proficiency; however, it asked for people that had experience in relational databases such as SQL. This is performing with a similarity score of 0.512.

These three scores being sclose together showed me that the chunks found were relevant to the query I input into the function. It also showed that it captured the meaning behind what I asked for.

---
<a id="5-analysis"></a>
## 5. Analysis Prompts and Chain

Build 3 analysis types, each with its own prompt (one iteration for 3 types) or 3 iterations for 1 type with its own prompt:

1. **Skill Gap Report:** Compare resume skills vs. JD requirements. Output matching skills, missing skills, and recommended actions.
2. **Keyword Alignment:** Extract key terms from a JD, check which appear in the resume, report a match rate.
3. **Fit Summary:** 3-4 sentence narrative assessment citing evidence from both documents.

You also need to wire up the LLM and a way to pass a specific JD + resume into each prompt.

**Required:** Document at least 3 prompt iterations total (across any analysis type) with rationale.

**Reminder:** Prompt design must be your own work (Tier 2 — AI prohibited for this step).

In [97]:
# ── Initialize LLM ──
from openai import OpenAI

if "OPENAI_API_KEY" in os.environ:
    del os.environ["OPENAI_API_KEY"]

# Load the .env file from the exact Colab path
load_dotenv("/content/.env", override=True)

# Check if the key loaded
print(os.getenv("OPENAI_API_KEY") is not None)

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("LLM initialized")

True
LLM initialized


In [98]:
print(documents[140])

Cybersecurity, Retail Tech, and Entertainment using advanced data analysis technologies. •	Develop a network of investors, CEOs, CISOs, and Heads of R&D across the West Coast that are interested in advising, investing, or buying from Israeli companies
•	Analyze and improve marketing materials and presentations for startups including data visualization and analytical insights
•	Consistently source relevant Israeli companies for Fortune 1000 companies based on current innovation initiatives and or investment theses

UCLA Computer Store                                                              
Lead Technician/Project Lead                                                                                      (January 2019-June 2020)
•	Troubleshot and repaired customers’ computers and devices
•	Set up UCLA Computer Store’s Custom Screen Protector Project
•	Led the training of 20 new employees in administrative and technical knowledge 

RESEARCH EXPERIENCE

Co-Mind Research Lab            

In [101]:
def skill_gap_analysis(question, k=5):

    # Retrieve
    query_embedding = embed_model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    # Build context
    context = "\n\n".join(results["documents"][0])

    # Build prompt
    prompt = f"""
Task: Explain what is similar and what is missing from this resume that is relevant for each job description and create a report from these results. Please include evidence for each similarity or gap and how to address each gap. Also include the job that they are comparing in the report.
Constraints: Use only the resume and job descriptions that are available and do not assume any information that is not clearly stated. If things are related (i.e. Tableau and Power BI), it can be indicated as similar. Make sure that missing content does not exist in the resume.
Context:
{context}

Question:
{question}

Answer:
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt},
                  {"role":"system" , "content": "You find the skill gaps of a person by comparing the job descriptions to resumes using only the documents that are provided"}],
        temperature=0.1
    )
    answer = response.choices[0].message.content
    return {
    "answer": answer,
    "context": context,
    "prompt": prompt
}

In [102]:
skill_gap_results= skill_gap_analysis("skills on this resume include SQL, Python, Tableau, Excel, PowerBI , Modeling, and other previous job experience")

print(f"Prompt: {skill_gap_results['prompt']}")

print(f"Response:{skill_gap_results['answer']}")

Prompt: 
Task: Explain what is similar and what is missing from this resume that is relevant for each job description and create a report from these results. Please include evidence for each similarity or gap and how to address each gap. Also include the job that they are comparing in the report.
Constraints: Use only the resume and job descriptions that are available and do not assume any information that is not clearly stated. If things are related (i.e. Tableau and Power BI), it can be indicated as similar. Make sure that missing content does not exist in the resume.
Context:
Business, Economics, Information Systems, Computer Science, or equivalent professional experience. Strong proficiency in Microsoft Excel, including intermediate to advanced formula skills, and familiarity with relational databases such as SQL. Basic understanding of statistical measures, techniques, and data visualization principles; experience with tools like Tableau or Power BI is preferred.

and Python progr

In [ ]:
# ── Analysis 1: Skill Gap Report (Prompts) ──
#Give me a analysis of the skill gap between the resume inputted and the job description (Baseline)
# Output for this prompt:
"""
Prompt:
Give me an analysis of the skill gap between the resume inputted and the job description.

Context:
Business, Economics, Information Systems, Computer Science, or equivalent professional experience. Strong proficiency in Microsoft Excel, including intermediate to advanced formula skills, and familiarity with relational databases such as SQL. Basic understanding of statistical measures, techniques, and data visualization principles; experience with tools like Tableau or Power BI is preferred.

and Python programming. Data architecture design and data preparation experience. Self-starter with strong problem-solving abilities and the ability to troubleshoot issues. Ability to communicate a coherent data story to business users. Must be proficient in MS Office (Word, Excel, PowerPoint). Excellent verbal, written, and interpersonal communication skills.

studies SKILLS • Programming: Proficient in MATLAB, Python, and SQL. Basic knowledge of R, C++, HTML 5, CSS, and JavaScript
•	Computer/Technical: Proficient in Tableau, Adobe Premiere Pro, Photoshop, Dreamweaver, FileMaker Pro, SolidWorks, Tableau Prep Builder, PowerBI, Microsoft Word, Excel, PowerPoint, Outlook, Keynote, and Pages.

of relevant experience is required. High school diploma or equivalent. Must be willing to partake in a comprehensive background check including a drug test in accordance with applicable laws. Proficiency with Domo or other Business Intelligence tools (i.e., Tableau, Power BI). Proficiency in SQL and Python programming. Data architecture design and data preparation experience.

visualization tools (Tableau and Google Looker expertise is a plus). Advanced proficiency with ETL and other data workflow tools (Alteryx, Matillion, etc). Proficient with common data languages (SQL, Python, R, etc). Experience using workflows for automation purposes is a plus. Proficiency with Ticketmaster / Archtics Data Platform preferred.

Question:
skills on this resume include SQL, Python, Tableau, Excel, PowerBI , Modeling, and other previous job experience

Answer:

Response:Based on the provided job description and the skills listed on the resume, here is an analysis of the skill gap:

### Skills Required by Job Description:
1. **Educational Background**: Business, Economics, Information Systems, Computer Science, or equivalent professional experience.
2. **Technical Skills**:
   - Strong proficiency in Microsoft Excel (intermediate to advanced formula skills).
   - Familiarity with relational databases (SQL).
   - Basic understanding of statistical measures, techniques, and data visualization principles.
   - Experience with data visualization tools (Tableau or Power BI preferred).
   - Python programming.
   - Data architecture design and data preparation experience.
   - Proficiency in MS Office (Word, Excel, PowerPoint).
   - Excellent verbal, written, and interpersonal communication skills.
   - Experience with ETL and data workflow tools (Alteryx, Matillion, etc.) is a plus.
   - Proficiency with Ticketmaster / Archtics Data Platform preferred.

### Skills Listed on the Resume:
1. **Programming**: Proficient in MATLAB, Python, and SQL; basic knowledge of R, C++, HTML5, CSS, and JavaScript.
2. **Computer/Technical Skills**: Proficient in Tableau, Adobe Premiere Pro, Photoshop, Dreamweaver, FileMaker Pro, SolidWorks, Tableau Prep Builder, Power BI, Microsoft Word, Excel, PowerPoint, Outlook, Keynote, and Pages.
3. **Other Relevant Experience**: Experience with data visualization tools (Tableau and Power BI), SQL, and Python programming.

### Skill Gap Analysis:
1. **Educational Background**: The resume does not explicitly mention the educational background. If the candidate does not have a degree in the specified fields, this could be a gap.

2. **Excel Proficiency**: The resume states proficiency in Excel, but it does not specify the level of proficiency (intermediate to advanced formula skills). This could be a potential gap if the candidate lacks advanced skills.

3. **Statistical Measures and Data Visualization**: The resume mentions experience with Tableau and Power BI, which aligns with the job description. However, it does not explicitly state a basic understanding of statistical measures and techniques, which is required.

4. **Data Architecture and Preparation**: The resume does not mention specific experience in data architecture design or data preparation, which is a requirement in the job description.

5. **Communication Skills**: While the resume lists various technical skills, it does not provide evidence of strong verbal, written, and interpersonal communication skills, which are emphasized in the job description.

6. **ETL and Data Workflow Tools**: The resume does not mention experience with ETL tools (like Alteryx or Matillion), which is a preferred qualification in the job description.

7. **Ticketmaster / Archtics Data Platform**: The resume does not indicate any experience with the Ticketmaster / Archtics Data Platform, which is preferred.

### Conclusion:
The candidate's resume shows a strong foundation in relevant technical skills such as SQL, Python, and data visualization tools (Tableau and Power BI). However, there are notable gaps in educational background clarity, advanced Excel skills, understanding of statistical measures, data architecture experience, communication skills, and familiarity with ETL tools and specific platforms like Ticketmaster. Addressing these gaps could enhance the candidate's alignment with the job requirements.
"""
#Based on the resume that was uploaded, please compare it to the job descriptions and identify which skills are matching and which ones are missing? Please provide advice on how to close the skills gap when there is one. Also when outputting the results please specify which company's job description is being analyzed. (Iteration 1)
#Output
'''
Prompt:
Based on the resume that was uploaded, please compare it to the job descriptions and identify which skills are matching and which ones are missing? Please provide advice on how to close the skills gap when there is one. Also when outputting the results please specify which company's job description is being analyzed.

Context:
Business, Economics, Information Systems, Computer Science, or equivalent professional experience. Strong proficiency in Microsoft Excel, including intermediate to advanced formula skills, and familiarity with relational databases such as SQL. Basic understanding of statistical measures, techniques, and data visualization principles; experience with tools like Tableau or Power BI is preferred.

and Python programming. Data architecture design and data preparation experience. Self-starter with strong problem-solving abilities and the ability to troubleshoot issues. Ability to communicate a coherent data story to business users. Must be proficient in MS Office (Word, Excel, PowerPoint). Excellent verbal, written, and interpersonal communication skills.

studies SKILLS • Programming: Proficient in MATLAB, Python, and SQL. Basic knowledge of R, C++, HTML 5, CSS, and JavaScript
•	Computer/Technical: Proficient in Tableau, Adobe Premiere Pro, Photoshop, Dreamweaver, FileMaker Pro, SolidWorks, Tableau Prep Builder, PowerBI, Microsoft Word, Excel, PowerPoint, Outlook, Keynote, and Pages.

of relevant experience is required. High school diploma or equivalent. Must be willing to partake in a comprehensive background check including a drug test in accordance with applicable laws. Proficiency with Domo or other Business Intelligence tools (i.e., Tableau, Power BI). Proficiency in SQL and Python programming. Data architecture design and data preparation experience.

visualization tools (Tableau and Google Looker expertise is a plus). Advanced proficiency with ETL and other data workflow tools (Alteryx, Matillion, etc). Proficient with common data languages (SQL, Python, R, etc). Experience using workflows for automation purposes is a plus. Proficiency with Ticketmaster / Archtics Data Platform preferred.

Question:
skills on this resume include SQL, Python, Tableau, Excel, PowerBI , Modeling, and other previous job experience

Answer:

Response:Based on the provided job descriptions and the skills listed in the resume, here’s a comparison of matching and missing skills, along with advice on how to close any skills gaps.

### Job Description Analysis

#### Company Job Description 1:
- **Required Skills:**
  - Proficiency in Microsoft Excel (intermediate to advanced)
  - Familiarity with relational databases (SQL)
  - Basic understanding of statistical measures and data visualization principles
  - Experience with tools like Tableau or Power BI
  - Python programming
  - Data architecture design and data preparation experience
  - Strong problem-solving abilities
  - Ability to communicate a coherent data story
  - Proficiency in MS Office (Word, Excel, PowerPoint)
  - Excellent communication skills

#### Company Job Description 2:
- **Required Skills:**
  - Proficiency with Domo or other Business Intelligence tools (Tableau, Power BI)
  - Proficiency in SQL and Python programming
  - Data architecture design and data preparation experience
  - Visualization tools expertise (Tableau and Google Looker)
  - Advanced proficiency with ETL and other data workflow tools (Alteryx, Matillion)
  - Proficiency with common data languages (SQL, Python, R)
  - Experience using workflows for automation

### Skills Matching from Resume:
- **Matching Skills:**
  - SQL: Proficient
  - Python: Proficient
  - Tableau: Proficient
  - Power BI: Proficient
  - Microsoft Excel: Proficient (though the level of proficiency is not specified)
  - Communication skills: Implied through previous job experience

### Skills Missing from Resume:
- **Missing Skills:**
  - Advanced Excel skills (intermediate to advanced formula skills)
  - Basic understanding of statistical measures and techniques
  - Data architecture design and data preparation experience (specific examples not provided)
  - Experience with ETL and data workflow tools (e.g., Alteryx, Matillion)
  - Familiarity with Google Looker
  - Experience with automation workflows
  - Proficiency in R (basic knowledge mentioned but not proficiency)

### Recommendations to Close Skills Gap:
1. **Advanced Excel Training:**
   - Enroll in an advanced Excel course focusing on formulas, pivot tables, and data analysis techniques.

2. **Statistical Analysis:**
   - Take an online course in statistics to understand basic statistical measures and techniques, which can be beneficial for data analysis.

3. **Data Architecture and Preparation:**
   - Gain hands-on experience or training in data architecture design and data preparation. Consider projects or internships that focus on these areas.

4. **ETL Tools:**
   - Learn about ETL tools like Alteryx or Matillion through online courses or tutorials. Practical experience with these tools can be gained through projects or simulations.

5. **Google Looker:**
   - Familiarize yourself with Google Looker by accessing its documentation and tutorials. Consider building sample dashboards to showcase your skills.

6. **Automation Workflows:**
   - Explore automation tools and workflows, possibly through Python libraries or platforms that support automation.

7. **R Programming:**
   - If time permits, consider taking a course in R to enhance your programming skills, especially for data analysis and visualization.

By focusing on these areas, you can strengthen your qualifications and better align your skills with the requirements of the job descriptions provided.
'''
#Explain what is similar and what is missing from this resume that is relevant to the job description and create a report from these results. Use only the resume and job descriptions that are available and do not assume any information that is not clearly stated. Please include evidence for each similarity or gap. (Iteration 2)
#Output
'''
Prompt:
Task: Explain what is similar and what is missing from this resume that is relevant to the job description and create a report from these results. Use only the resume and job descriptions that are available and do not assume any information that is not clearly stated. Please include evidence for each similarity or gap.
Context:
Business, Economics, Information Systems, Computer Science, or equivalent professional experience. Strong proficiency in Microsoft Excel, including intermediate to advanced formula skills, and familiarity with relational databases such as SQL. Basic understanding of statistical measures, techniques, and data visualization principles; experience with tools like Tableau or Power BI is preferred.

and Python programming. Data architecture design and data preparation experience. Self-starter with strong problem-solving abilities and the ability to troubleshoot issues. Ability to communicate a coherent data story to business users. Must be proficient in MS Office (Word, Excel, PowerPoint). Excellent verbal, written, and interpersonal communication skills.

studies SKILLS • Programming: Proficient in MATLAB, Python, and SQL. Basic knowledge of R, C++, HTML 5, CSS, and JavaScript
•	Computer/Technical: Proficient in Tableau, Adobe Premiere Pro, Photoshop, Dreamweaver, FileMaker Pro, SolidWorks, Tableau Prep Builder, PowerBI, Microsoft Word, Excel, PowerPoint, Outlook, Keynote, and Pages.

of relevant experience is required. High school diploma or equivalent. Must be willing to partake in a comprehensive background check including a drug test in accordance with applicable laws. Proficiency with Domo or other Business Intelligence tools (i.e., Tableau, Power BI). Proficiency in SQL and Python programming. Data architecture design and data preparation experience.

visualization tools (Tableau and Google Looker expertise is a plus). Advanced proficiency with ETL and other data workflow tools (Alteryx, Matillion, etc). Proficient with common data languages (SQL, Python, R, etc). Experience using workflows for automation purposes is a plus. Proficiency with Ticketmaster / Archtics Data Platform preferred.

Question:
skills on this resume include SQL, Python, Tableau, Excel, PowerBI , Modeling, and other previous job experience

Answer:

Response:### Report on Resume Analysis Against Job Description

#### Similarities

1. **Programming Skills**:
   - **Job Description Requirement**: Proficiency in Python programming.
   - **Resume Evidence**: The resume lists proficiency in Python, which aligns with the job requirement.

2. **SQL Proficiency**:
   - **Job Description Requirement**: Familiarity with relational databases such as SQL.
   - **Resume Evidence**: The resume indicates proficiency in SQL, meeting the job's expectations.

3. **Data Visualization Tools**:
   - **Job Description Requirement**: Experience with tools like Tableau or Power BI.
   - **Resume Evidence**: The resume mentions proficiency in both Tableau and Power BI, fulfilling this requirement.

4. **Microsoft Excel**:
   - **Job Description Requirement**: Strong proficiency in Microsoft Excel, including intermediate to advanced formula skills.
   - **Resume Evidence**: The resume states proficiency in Microsoft Excel, which is a direct match.

5. **Communication Skills**:
   - **Job Description Requirement**: Excellent verbal, written, and interpersonal communication skills.
   - **Resume Evidence**: While not explicitly stated, the inclusion of various technical skills and tools suggests a level of communication proficiency necessary for conveying data insights.

#### Gaps

1. **Statistical Measures and Techniques**:
   - **Job Description Requirement**: Basic understanding of statistical measures, techniques, and data visualization principles.
   - **Resume Evidence**: The resume does not mention any experience or knowledge in statistical measures or techniques, indicating a gap.

2. **Data Architecture Design and Preparation**:
   - **Job Description Requirement**: Data architecture design and data preparation experience.
   - **Resume Evidence**: The resume does not provide any evidence of experience in data architecture design or data preparation, which is a critical requirement.

3. **ETL and Data Workflow Tools**:
   - **Job Description Requirement**: Advanced proficiency with ETL and other data workflow tools (e.g., Alteryx, Matillion).
   - **Resume Evidence**: The resume does not mention any experience with ETL tools or data workflow automation, representing a significant gap.

4. **Experience with Domo or Other Business Intelligence Tools**:
   - **Job Description Requirement**: Proficiency with Domo or other Business Intelligence tools.
   - **Resume Evidence**: The resume does not indicate any experience with Domo, which is specifically mentioned in the job description.

5. **Automation Workflows**:
   - **Job Description Requirement**: Experience using workflows for automation purposes is a plus.
   - **Resume Evidence**: There is no mention of experience with automation workflows, which could enhance the candidate's profile.

6. **Ticketmaster / Archtics Data Platform**:
   - **Job Description Requirement**: Proficiency with Ticketmaster / Archtics Data Platform preferred.
   - **Resume Evidence**: The resume does not reference any experience with Ticketmaster or Archtics, which is a preferred qualification.

### Conclusion

The resume demonstrates strong alignment with several key skills required by the job description, particularly in programming, SQL, data visualization, and Microsoft Excel. However, there are notable gaps in statistical knowledge, data architecture experience, ETL tools, and specific business intelligence tools like Domo. Addressing these gaps could significantly enhance the candidate's suitability for the position.
'''

#Task: Explain what is similar and what is missing from this resume that is relevant for each job description and create a report from these results. Please include evidence for each similarity or gap and how to address each gap. Also include the job that they are comparing in the report.
#Constraints: Use only the resume and job descriptions that are available and do not assume any information that is not clearly stated. If things are related (i.e. Tableau and Power BI), it can be indicated as similar. Make sure that missing content does not exist in the resume. (Iteration 3)
#Output
'''
Prompt:
Task: Explain what is similar and what is missing from this resume that is relevant for each job description and create a report from these results. Please include evidence for each similarity or gap and how to address each gap. Also include the job that they are comparing in the report.
Constraints: Use only the resume and job descriptions that are available and do not assume any information that is not clearly stated. If things are related (i.e. Tableau and Power BI), it can be indicated as similar. Make sure that missing content does not exist in the resume.
Context:
Business, Economics, Information Systems, Computer Science, or equivalent professional experience. Strong proficiency in Microsoft Excel, including intermediate to advanced formula skills, and familiarity with relational databases such as SQL. Basic understanding of statistical measures, techniques, and data visualization principles; experience with tools like Tableau or Power BI is preferred.

and Python programming. Data architecture design and data preparation experience. Self-starter with strong problem-solving abilities and the ability to troubleshoot issues. Ability to communicate a coherent data story to business users. Must be proficient in MS Office (Word, Excel, PowerPoint). Excellent verbal, written, and interpersonal communication skills.

studies SKILLS • Programming: Proficient in MATLAB, Python, and SQL. Basic knowledge of R, C++, HTML 5, CSS, and JavaScript
•	Computer/Technical: Proficient in Tableau, Adobe Premiere Pro, Photoshop, Dreamweaver, FileMaker Pro, SolidWorks, Tableau Prep Builder, PowerBI, Microsoft Word, Excel, PowerPoint, Outlook, Keynote, and Pages.

of relevant experience is required. High school diploma or equivalent. Must be willing to partake in a comprehensive background check including a drug test in accordance with applicable laws. Proficiency with Domo or other Business Intelligence tools (i.e., Tableau, Power BI). Proficiency in SQL and Python programming. Data architecture design and data preparation experience.

visualization tools (Tableau and Google Looker expertise is a plus). Advanced proficiency with ETL and other data workflow tools (Alteryx, Matillion, etc). Proficient with common data languages (SQL, Python, R, etc). Experience using workflows for automation purposes is a plus. Proficiency with Ticketmaster / Archtics Data Platform preferred.

Question:
skills on this resume include SQL, Python, Tableau, Excel, PowerBI , Modeling, and other previous job experience

Answer:

Response:# Resume and Job Description Comparison Report

## Job Description Overview
The job descriptions provided emphasize the following key requirements:
1. Educational background in Business, Economics, Information Systems, Computer Science, or equivalent professional experience.
2. Strong proficiency in Microsoft Excel, including intermediate to advanced formula skills.
3. Familiarity with relational databases such as SQL.
4. Basic understanding of statistical measures, techniques, and data visualization principles.
5. Experience with data visualization tools like Tableau or Power BI.
6. Proficiency in Python programming.
7. Experience in data architecture design and data preparation.
8. Strong problem-solving abilities and troubleshooting skills.
9. Ability to communicate data insights effectively to business users.
10. Proficiency in MS Office (Word, Excel, PowerPoint).
11. Excellent verbal, written, and interpersonal communication skills.
12. Experience with ETL and other data workflow tools (Alteryx, Matillion, etc.) is a plus.
13. Proficiency with Domo or other Business Intelligence tools is preferred.
14. Experience with Ticketmaster / Archtics Data Platform is preferred.

## Resume Skills Overview
The resume lists the following skills:
- SQL
- Python
- Tableau
- Excel
- Power BI
- Modeling
- Other previous job experience

## Similarities and Gaps Analysis

### Similarities
1. **SQL Proficiency**:
   - **Evidence**: The resume lists SQL as a skill, which aligns with the job descriptions that require proficiency in SQL.

2. **Python Programming**:
   - **Evidence**: Python is mentioned in both the resume and job descriptions, indicating a match in programming skills.

3. **Data Visualization Tools**:
   - **Evidence**: The resume includes Tableau and Power BI, which are specifically mentioned in the job descriptions as preferred tools.

4. **Microsoft Excel**:
   - **Evidence**: Excel is listed in the resume, matching the job descriptions that require strong proficiency in Excel.

### Gaps
1. **Educational Background**:
   - **Gap**: The resume does not specify the educational background (e.g., Business, Economics, Information Systems, Computer Science).
   - **Addressing the Gap**: Include relevant educational qualifications or equivalent professional experience in the resume.

2. **Intermediate to Advanced Excel Skills**:
   - **Gap**: While Excel is mentioned, there is no indication of intermediate to advanced formula skills.
   - **Addressing the Gap**: Specify any advanced Excel skills or relevant projects that demonstrate proficiency with formulas.

3. **Statistical Measures and Techniques**:
   - **Gap**: The resume does not mention any understanding of statistical measures or techniques.
   - **Addressing the Gap**: Include any coursework, certifications, or experience related to statistical analysis.

4. **Data Architecture Design and Data Preparation**:
   - **Gap**: The resume lacks specific mention of experience in data architecture design and data preparation.
   - **Addressing the Gap**: Highlight any relevant projects or roles that involved data architecture or preparation tasks.

5. **Problem-Solving Abilities**:
   - **Gap**: There is no explicit mention of problem-solving abilities or troubleshooting skills.
   - **Addressing the Gap**: Provide examples of past experiences where problem-solving was key to success.

6. **Communication Skills**:
   - **Gap**: The resume does not detail verbal, written, and interpersonal communication skills.
   - **Addressing the Gap**: Include examples of communication skills in action, such as presentations or team collaborations.

7. **ETL and Data Workflow Tools**:
   - **Gap**: The resume does not mention experience with ETL tools or data workflow tools like Alteryx or Matillion.
   - **Addressing the Gap**: If applicable, include any experience with ETL processes or tools used in previous roles.

8. **Experience with Domo or Ticketmaster / Archtics Data Platform**:
   - **Gap**: The resume does not mention familiarity with Domo or the Ticketmaster / Archtics Data Platform.
   - **Addressing the Gap**: If applicable, include any experience with these platforms or similar tools.

## Conclusion
The resume demonstrates a strong foundation in key technical skills relevant to the job descriptions, particularly in SQL, Python, and data visualization tools. However, there are several gaps related to educational background, advanced Excel skills, statistical knowledge, and specific experiences that need to be addressed to enhance the alignment with the job requirements. By incorporating the suggested improvements, the candidate can present a more compelling case for their suitability for the roles described
'''

### Prompt Iteration Log

Document at least 3 total iterations across any of the analysis types.

**Baseline:** When I put the initial prompt for the skills gap analysis, it performed well. It was able to show a summarized version of the skills from the job based on the description. It was really able to take the exact skills from the resume and apply it to the model in its own sections. The conclusion could be more detailed but is a good summary. The last thing that is missing is indicating which job description it is showing.

**Iteration 1:** The first analysis that I am testing the iterations on is the Skills Gap Analysis. The first itteration changed by adding more details into the prompt and trying to make it provide ways to improve the skill gap instead of just outputting it for one description. By adding more details to the prompt, it made the results more task oriented. It improved because it was able to identify the gaps in more than one description and also provided a way to close the skills gap for all of the the job descriptions in one list.

**Iteration 2:** For the second iteration, I conducted it on the Skills Gap Analysis again. This time I very clearly stated that it was a task rather than keeping it as a prompt. I wrote the prompt this way because this makes the model understand its purpose instead of the model trying to figure out what the its task is from the prompt. Another difference that I have had was that I started to add specific constraints. This limits the information that the model RAG model can retrieve and then applied to the results. I did this because of the fact that a RAG model can hallucinate depending on how much creativity it is given. With this I decided to limit what information it can pull and then make sure it is accurate. Another thing that I added was that the results should be output as a report. This was good because it dictated the way the output is formatted. Although, I missed the part about addressing the skill gap in the prompt. This changed the way that we output was generated. The ability to retrieve more accurately has improved significantly and the model was able to stick closer to the model because of the fact that restrictions were in place.

**Iteration 3:** [Which analysis? What changed? Why? What improved?]
For this iteration, I re-iterated the Skills Gap Analysis. In this point, I kept adding more details while keeping a similar structure to iteration 2. This changed the overall output beccause it finalized it as more of a report. What mainly changed is that I added some more elements to the report to make sure that we get the most information from the RAG model as possible. This improved the thoroughness of the report and how it looked with the formatting. This looked more like an official report and could easily be looked at for understandable information.

In [112]:
# ── Analysis 2: Keyword Alignment ──
def keyword_alignment_analysis(question, k=10):

    # Retrieve
    query_embedding = embed_model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    # Build context
    context = "\n\n".join(results["documents"][0])

    # Build prompt
    prompt = f"""
Task: Based on the resume, job descriptions, and keyword taken in the question to create keyword alignment analysis that compares the keywords in the job description to the resume.
Constraints: Use only the resume, job descriptions, and keywords that are available and do not assume any information that is not clearly stated.
Output: The output should include they keyword, Whether it includes any of the following direct match, semantic match, or missing, and the evidence from the job description and resume. Also show the keyword alignment percentage for each word along with the general. The next piece of the output should be a conclusion for each key word. The final output should be a report with the general conclusions.

Context:
{context}

Question:
{question}

Answer:
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt},
                  {"role":"system" , "content": "You find the relevant positions by comparing keywords in the job description to the resume"}],
        temperature=0.2
    )
    answer = response.choices[0].message.content
    return {
    "answer": answer,
    "context": context,
    "prompt": prompt
}

In [114]:
keyword_results=keyword_alignment_analysis("sql data analysis tableau analytics strategy insights consumers innovation modeling corporate development visualization sustainability advance matchmaking")
print(keyword_results['prompt'])
print(keyword_results['answer'])


Task: Based on the resume, job descriptions, and keyword taken in the question to create keyword alignment analysis that compares the keywords in the job description to the resume.
Constraints: Use only the resume, job descriptions, and keywords that are available and do not assume any information that is not clearly stated.
Output: The output should include they keyword, Whether it includes any of the following direct match, semantic match, or missing, and the evidence from the job description and resume. Also show the keyword alignment percentage for each word along with the general. The next piece of the output should be a conclusion for each key word. The final output should be a report with the general conclusions. 

Context:
most sophisticated insights from their data in a value-oriented manner. From analytics foundation to analytics innovation initiatives, we offer a comprehensive range of services that help businesses succeed with data.

with opportunities to grow into data en

In [ ]:
# ── Analysis 3: Fit Summary ──
def fit_summary_analysis(question, k=5):

    # Retrieve
    query_embedding = embed_model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=len(question)
    )

    # Build context
    context = "\n\n".join(results["documents"][0])

    # Build prompt
    prompt = f"""

Task: Based on the resume and job descriptions create

Context:
{context}

Question:
{question}

Answer:
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt},
                  {"role":"system" , "content": "You evaluate if a candidate's fit for the role based on the job descriptions and the resume of the candidate."}],
        temperature=0.1
    )
    answer = response.choices[0].message.content
    return {
    "answer": answer,
    "sources": sources,
    "context": context,
    "prompt": full_prompt
}

---
<a id="6-comparison"></a>
## 6. Zero-shot vs. Few-shot Comparison

Pick one of your 3 analysis types. Create a few-shot version by adding 1-2 example input/output pairs to the prompt. Run both versions on the same JD and compare outputs.

**Reminder:** You must write the few-shot examples yourself (Tier 2).

In [ ]:
# ── Few-shot version of your chosen analysis ──
# Based on the following job description, identify whether the job is match from the resume.

#Job Description: (insert job description here)
#Match: Yes because ....

#Job Description: (insert job description here)
#Match: No because ....

In [ ]:
# ── Run both on the same JD, display side by side ──


### Zero-shot vs. Few-shot Analysis

**Which analysis type did you compare?**

**Which performed better?**

**Why? (use specific examples from the outputs above)**

---
<a id="7-evaluation"></a>
## 7. Evaluation

Run all 3 analysis types on your **top 3 target JDs** (9 total analyses).

For each, score:
- **Retrieval relevance:** Did it pull the right JD sections? (Yes/Partial/No)
- **Skill identification accuracy:** Are identified skills/gaps correct? (count correct vs. incorrect)
- **Actionability:** Are recommendations specific and useful? (1-5)
- **Faithfulness:** Does output stick to document content? (Faithful/Partial/Hallucinated)

**Reminder:** Evaluation must be your own work (Tier 2 — AI prohibited).

In [ ]:
# ── Run 9 analyses (3 JDs x 3 analysis types) ──


In [ ]:
# ── Summarize evaluation results ──


### Evaluation Analysis

**Which analysis type worked best?**

**Which JDs produced the best/worst results? Why?**

**Where did the system hallucinate or produce inaccurate results?**

**What would you improve?**

---

## Next Steps

1. Build your Streamlit app (`streamlit_app.py`) using the pipeline from this notebook
2. Write your Technical Manager Memo (`memo.md`)
3. Complete your AI Usage Log (`ai_log.md`)
4. Verify GitHub repository structure and commit count

---
*BSAN 6200 | Spring 2026 | Assignment 5 — Option B*